In [1]:
"""
Примеры кода к Лекции 5.6: Обработка ошибок — как сделать агента устойчивым

Этот модуль демонстрирует:
1. Деградация в узле — частичный результат вместо краша
2. Retry с экспоненциальным backoff внутри узла
3. RetryPolicy — встроенный retry LangGraph на уровне узла
4. Attempt-aware узлы — runtime.execution_info
5. Fallback-ребро — запасной путь на уровне графа
6. ToolNode с обработкой ошибок
7. Глобальные лимиты — recursion_limit и step_timeout
"""

'\nПримеры кода к Лекции 5.6: Обработка ошибок — как сделать агента устойчивым\n\nЭтот модуль демонстрирует:\n1. Деградация в узле — частичный результат вместо краша\n2. Retry с экспоненциальным backoff внутри узла\n3. RetryPolicy — встроенный retry LangGraph на уровне узла\n4. Attempt-aware узлы — runtime.execution_info\n5. Fallback-ребро — запасной путь на уровне графа\n6. ToolNode с обработкой ошибок\n7. Глобальные лимиты — recursion_limit и step_timeout\n'

In [2]:
import random
import time
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.types import RetryPolicy

from llm_config import check_api_key, get_llm

In [3]:
if not check_api_key():
    raise ValueError("API key is not set")

## Деградация в узле

In [4]:
class State(TypedDict):
    query: str
    search_results: str
    answer: str

def search_node(state: State) -> dict:
    """Узел поиска с деградацией при ошибке."""
    try:
        # Имитация сбоя внешнего API
        if random.random() < 0.7:
            raise ConnectionError("Search API unavailable")
        return {"search_results": f"Результаты по '{state['query']}': ..."}
    except ConnectionError as e:
        print(f"  [degradation] Поиск недоступен: {e}")
        return {"search_results": "Поиск недоступен. Отвечаю из своих знаний."}

def answer_node(state: State) -> dict:
    """Генерирует ответ на основе результатов поиска."""
    llm = get_llm()
    msg = llm.invoke(
        f"Вопрос: {state['query']}\nКонтекст: {state['search_results']}\n"
        f"Дай краткий ответ."
    )
    return {"answer": msg.content}

builder = StateGraph(State)
builder.add_node("search", search_node)
builder.add_node("answer", answer_node)
builder.add_edge(START, "search")
builder.add_edge("search", "answer")
builder.add_edge("answer", END)

graph = builder.compile()
result = graph.invoke({"query": "Что такое LangGraph?"})
print(f"  Поиск: {result['search_results'][:80]}...")
print(f"  Ответ: {result['answer'][:150]}...")

  Поиск: Результаты по 'Что такое LangGraph?': ......
  Ответ: LangGraph — это библиотека для построения **состоянийных графов** и **агентных workflow** поверх LangChain. Она помогает создавать сложные цепочки дей...


## Retry с backoff внутри узла

In [5]:
class State(TypedDict):
    query: str
    result: str

call_count = 0

def flaky_api_node(state: State) -> dict:
    """Узел с retry при транзиентных ошибках."""
    global call_count
    max_retries = 3
    delay = 0.5

    for attempt in range(max_retries):
        try:
            call_count += 1
            # Имитация: первые 2 вызова — ошибка, третий — успех
            if call_count < 3:
                raise ConnectionError(f"API timeout (attempt {attempt + 1})")
            return {"result": f"Успех на попытке {attempt + 1}"}
        except ConnectionError as e:
            print(f"  [retry] Попытка {attempt + 1}/{max_retries}: {e}")
            if attempt < max_retries - 1:
                jitter = random.uniform(0.5, 1.5)
                sleep_time = delay * (2**attempt) * jitter
                print(f"  [retry] Ожидание {sleep_time:.1f}с...")
                time.sleep(min(sleep_time, 1.0))  # Ограничим для примера

    # Деградация при исчерпании попыток
    print("  [retry] Все попытки исчерпаны, деградация")
    return {"result": "Сервис временно недоступен"}

builder = StateGraph(State)
builder.add_node("api_call", flaky_api_node)
builder.add_edge(START, "api_call")
builder.add_edge("api_call", END)

graph = builder.compile()
result = graph.invoke({"query": "test"})
print(f"  Результат: {result['result']}")

  [retry] Попытка 1/3: API timeout (attempt 1)
  [retry] Ожидание 0.4с...


  [retry] Попытка 2/3: API timeout (attempt 2)
  [retry] Ожидание 0.6с...


  Результат: Успех на попытке 3


## RetryPolicy — встроенный retry LangGraph

In [6]:
class State(TypedDict):
    query: str
    result: str

attempt_counter = {"count": 0}

def unreliable_node(state: State) -> dict:
    """Узел, который падает первые 2 раза."""
    attempt_counter["count"] += 1
    attempt = attempt_counter["count"]
    print(f"  [node] Попытка #{attempt}")

    if attempt < 3:
        raise ConnectionError(f"Transient error on attempt {attempt}")

    return {"result": f"Успех на попытке {attempt}"}

builder = StateGraph(State)
builder.add_node(
    "unreliable",
    unreliable_node,
    retry_policy=RetryPolicy(
        max_attempts=5,
        initial_interval=0.3,
        backoff_factor=2.0,
        jitter=True,
    ),
)
builder.add_edge(START, "unreliable")
builder.add_edge("unreliable", END)

graph = builder.compile()
result = graph.invoke({"query": "test"})
print(f"  Результат: {result['result']}")

  [node] Попытка #1


  [node] Попытка #2


  [node] Попытка #3
  Результат: Успех на попытке 3


## Attempt-aware узлы — runtime.execution_info

In [7]:
class State(TypedDict):
    query: str
    result: str

def smart_node(state: State, runtime) -> dict:
    """Узел, который знает номер попытки."""
    info = runtime.execution_info
    attempt = info.node_attempt
    print(f"  [node] Попытка #{attempt}")

    if attempt == 1:
        # Первая попытка — пробуем основную модель
        print("  [node] Основная модель (может упасть)...")
        raise ConnectionError("Primary model timeout")
    else:
        # Повторная попытка — fallback на простую логику
        print("  [node] Fallback: простой ответ без модели")
        return {"result": f"Fallback ответ на '{state['query']}'"}

builder = StateGraph(State)
builder.add_node(
    "smart",
    smart_node,
    retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.2),
)
builder.add_edge(START, "smart")
builder.add_edge("smart", END)

graph = builder.compile()
result = graph.invoke({"query": "Что такое LangGraph?"})
print(f"  Результат: {result['result']}")

  [node] Попытка #1
  [node] Основная модель (может упасть)...


  [node] Попытка #2
  [node] Fallback: простой ответ без модели
  Результат: Fallback ответ на 'Что такое LangGraph?'


## Fallback-ребро

In [8]:
class State(TypedDict):
    query: str
    result: str
    error: str

def primary_node(state: State) -> dict:
    """Основной узел — сложная обработка, может упасть."""
    print("  [primary] Сложная RAG-цепочка...")
    # Имитация ошибки
    return {
        "error": "RAG pipeline failed: vector store unavailable",
        "result": "",
    }

def fallback_node(state: State) -> dict:
    """Запасной узел — простая, но надёжная обработка."""
    print("  [fallback] Простой ответ без RAG")
    llm = get_llm()
    msg = llm.invoke(f"Ответь кратко на вопрос: {state['query']}")
    return {"result": msg.content, "error": ""}

def finalize_node(state: State) -> dict:
    """Финализация — работает в обоих случаях."""
    source = "fallback" if "fallback" in state.get("result", "") else "primary"
    print(f"  [finalize] Источник: {source}")
    return state

def route_on_error(state: State) -> str:
    """Маршрутизатор: есть ошибка → fallback, нет → finalize."""
    if state.get("error"):
        print(f"  [router] Ошибка: {state['error'][:50]}... → fallback")
        return "fallback"
    print("  [router] OK → finalize")
    return "finalize"

builder = StateGraph(State)
builder.add_node("primary", primary_node)
builder.add_node("fallback", fallback_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START, "primary")
builder.add_conditional_edges(
    "primary", route_on_error, {"fallback": "fallback", "finalize": "finalize"}
)
builder.add_edge("fallback", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile()
result = graph.invoke({"query": "Что такое LangGraph?"})
print(f"  Результат: {result['result'][:150]}...")

  [primary] Сложная RAG-цепочка...
  [router] Ошибка: RAG pipeline failed: vector store unavailable... → fallback
  [fallback] Простой ответ без RAG


  [finalize] Источник: primary
  Результат: LangGraph — это фреймворк для построения приложений на базе LLM в виде графа состояний и шагов, где можно задавать циклы, ветвления и сложную логику в...


## ToolNode с обработкой ошибок

In [9]:
def example_tool_error_handling():
    """
    ToolNode с handle_tool_errors=True: ошибка инструмента
    возвращается модели как ToolMessage, а не крашит граф.
    """
    @tool
    def divide(a: float, b: float) -> str:
        """Делит a на b."""
        if b == 0:
            raise ValueError("Деление на ноль!")
        return str(a / b)

    @tool
    def search_db(query: str) -> str:
        """Поиск в базе данных."""
        raise ConnectionError("Database connection refused")

    # По умолчанию (>=1.0.1) — ловит только ToolInvocationError (валидация)
    tool_node_default = ToolNode([divide, search_db])
    print("  Дефолт: ловит только ошибки валидации аргументов")

    # Явный catch-all — ловит все ошибки и возвращает модели
    tool_node_catchall = ToolNode([divide, search_db], handle_tool_errors=True)
    print("  handle_tool_errors=True: ловит ВСЕ ошибки")

    # Кастомный обработчик — форматирует ошибку
    def format_error(e: Exception) -> str:
        return f"Инструмент недоступен: {type(e).__name__}. Попробуй другой подход."

    tool_node_custom = ToolNode(
        [divide, search_db], handle_tool_errors=format_error
    )
    print("  Кастомный обработчик: форматирует ошибку для модели")

## Глобальные лимиты

In [10]:
class State(TypedDict):
    counter: int

def increment(state: State) -> dict:
    new_val = state["counter"] + 1
    print(f"  [node] counter = {new_val}")
    return {"counter": new_val}

def should_continue(state: State) -> str:
    # Намеренно никогда не завершаемся — для демонстрации лимита
    return "continue"

builder = StateGraph(State)
builder.add_node("increment", increment)
builder.add_edge(START, "increment")
builder.add_conditional_edges(
    "increment", should_continue, {"continue": "increment", "stop": END}
)

graph = builder.compile()

try:
    result = graph.invoke({"counter": 0}, {"recursion_limit": 8})
except GraphRecursionError as e:
    print(f"  GraphRecursionError: граф остановлен на лимите")
    print(f"  Это защита от бесконечных циклов и неконтролируемых расходов")

  [node] counter = 1
  [node] counter = 2
  [node] counter = 3
  [node] counter = 4
  [node] counter = 5
  [node] counter = 6
  [node] counter = 7
  [node] counter = 8
  GraphRecursionError: граф остановлен на лимите
  Это защита от бесконечных циклов и неконтролируемых расходов
